In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")


In [18]:
query = "두쫀쿠가 뭐지?"

In [13]:
response = model.invoke(query)
response

AIMessage(content="**두쫀쿠**는 **'두꺼운 쫀득한 쿠키'**의 줄임말입니다.\n\n*   **두**: **두꺼운**\n*   **쫀**: **쫀득한**\n*   **쿠**: **쿠키**\n\n말 그대로 두께감이 있으면서 속은 쫀득쫀득하고 부드러운 식감을 가진 쿠키를 의미합니다. 요즘 홈베이킹이나 수제 쿠키 가게에서 매우 인기 있는 종류의 쿠키를 지칭할 때 많이 사용됩니다.\n\n주로 겉은 바삭하지만 속은 촉촉하고 쫀득한 식감이 특징이며, 초코칩 두쫀쿠, 황치즈 두쫀쿠, 말차 두쫀쿠 등 다양한 맛으로 만들어집니다.", additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--95d1de8e-0c23-4a54-a307-5be96bb6931e-0', usage_metadata={'input_tokens': 8, 'output_tokens': 1122, 'total_tokens': 1130, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 963}})

In [14]:
from langchain_community.tools import DuckDuckGoSearchResults 

search = DuckDuckGoSearchResults(results_separator=';\n')
docs = search.invoke(query)

print(docs)

snippet: 나만 두쫀쿠가 뭔지 몰랐어? 공무원 · T*****. 작성일어제 조회수91 댓글7.인터넷에 두쫀쿠가 나온적이 없는데? 작성일어제 좋아요수좋아요., title: 블라인드 | 블라블라: 나만 두쫀쿠가 뭔지 몰랐어?, link: https://www.teamblind.com/kr/post/나만-두쫀쿠가-뭔지-몰랐어-mm4e5nk3;
snippet: 두쫀쿠가 그정도인가. 먹어봤을때 맛은 괜찮긴한데 난리칠 정돈 아닌거 같음.메이플 이사람들 뭐지 ㄹㅇ [134]. 리니지M 재료캘라고 5만원짜리 암기 하나 사왔는데 죄책감 드네.., title: 로스트아크 인벤 : 두쫀쿠가 그정도인가 - 로스트아크 인벤 자유 게시판, link: https://www.inven.co.kr/board/lostark/6271/3168444;
snippet: 두쫀쿠가 자영업계에 가져온 변화. 당근이지당근 10 시간 전., title: 두쫀쿠가 자영업계에 가져온 변화 - DogDrip.Net 개드립, link: https://www.dogdrip.net/679012262;
snippet: 김지혜는 1월 15일 자신의 사회관계망서비스 계정에 " 뭐지 이건 왜 맛없지 ㅋㅋ"라는 글과 함께 사진을 게재했다. 사진 속엔 몇 번 베어먹은 흔적이 남은 두쫀쿠가 담겨 있다., title: ‘박준형 ’ 김지혜, 품절대란 두쫀쿠 먹고 실망 “왜 맛없지” - 손에 잡히는..., link: https://www.newsen.com/news_view.php?uid=202601161349346110


### 🔎 웹 검색 도구

In [15]:
from langchain_core.tools import tool
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper
from langchain_community.tools import DuckDuckGoSearchResults

@tool
def get_web_search(query: str, search_period: str='w', region: str='kr-kr') -> str:
    """
    특정 지역과 기간을 설정하여 웹 검색(뉴스)을 수행합니다.

    Args:
        query (str): 검색어
        search_period (str): 검색 기간. 'd'(일간), 'w'(주간), 'm'(월간), 'y'(연간) 중 선택
        region (str): 검색 지역 코드. 한국('kr-kr'), 미국('us-en'), 일본('jp-jp'), 영국('uk-en') 등
    
    Returns:
        str: 검색된 뉴스 결과 리스트
    """
    # 1. 전달받은 region과 search_period로 API 설정
    wrapper = DuckDuckGoSearchAPIWrapper(
        region=region,
        time=search_period
    )

    print('\n----- WEB SEARCH (News) -----')
    print(f"QUERY  : {query}")
    print(f"REGION : {region}")
    print(f"PERIOD : {search_period}")

    search = DuckDuckGoSearchResults(
        api_wrapper=wrapper,
        backend="news",
        results_separator=';\n'
    )

    searched = search.invoke(query)
    return searched

In [16]:
from langchain.agents import create_agent

# 도구들을 tools 리스트에 추가
tools = [get_web_search,]

# 에이전트 생성
agent = create_agent(
    model,
    tools=tools,
    system_prompt="너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."
)

In [19]:
query

'두쫀쿠가 뭐지?'

In [20]:
# Run the agent
result = agent.invoke(
    {"messages": [{"role": "user", "content": query}]}
)


----- WEB SEARCH (News) -----
QUERY  : 두쫀쿠
REGION : kr-kr
PERIOD : w


In [21]:
result

{'messages': [HumanMessage(content='두쫀쿠가 뭐지?', additional_kwargs={}, response_metadata={}, id='3d829187-c30a-45c3-bff3-d754c49413bc'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_web_search', 'arguments': '{"query": "\\ub450\\ucac0\\ucfe0"}'}, '__gemini_function_call_thought_signatures__': {'e3e0e53a-054f-4488-b6f3-116ac12b4360': 'Ct4DAXLI2nxSg1g0U7YR7Urv9tjV00W3o4F1QrRCNYpq+Mzljsb1/leMnnwsrCip6VPNgBMh06YFTYGf3OIiWS7ApM621Dpoz1wu07HLTX/suJPsNHvKS6UNqCvi+JywxxNuvNurM0ykWbpY2cDSCzxpaz5vry5ilC/u4MhO+hZUaIgi00sI8qvRcI26+fAaMGurnDjZT3TuC11e1q/6xetAniGd6p+kkNReB3XJLq/CxhGafpbLb1hSOvyItzOapRVp5zVrrbtgyNmGr2E7wKBYPO08USmZAAD56dNCBCF29SM0S6TtP7AFIajZBK6jxgneJUFrdZ1hIuY/5+uKiTViuxYu4wT7eDpD5kpu32QMrr1HyZ9xhs5A9p2IZROTmfIyLFrwVkXVyTS848C4t1fT8+qUYGY8s95N+NlfBOD8FMtNlz2GUiJPHrse8Fcc6aGyLe1XuZu2kHGKmGdQCZwNGq1PcLtqt2lCA230ijXxaRJPuaYtU/eMgUpkEdAu37LqIpPK7SNnSFzeECIKOv49eYnzCbNtknV+HmkmofLm6KU6aJyYSzoDeNFPRoxcWKjoOECTlnWoi5lNqrdVPtH8BBWw1t6n6cYxi7tzME+ryBCUeH2f1YAWJ6m2q